#  ⭐ `fat_medicamentos`


In [10]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings ##, normalize_duration, expandir_gravidade_wide
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [11]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO'],
      dtype='object')

In [3]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
0,BR-ANVISA-300000004,Suspeito,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,None,None,None,None,None,250 milligram (mg),6 hora,None,4 dia,20181124,None,None,infusão intravenosa,None,5833018,2025-11-17
1,BR-ANVISA-300000005,Suspeito,Paracemol,Paracetamol,N02BE,None,None,None,Retirada do medicamento,None,None,None,None,None,None,None,20181122,20181122,None,oral,None,None,2025-11-17


# 🥈 Silver

normalized data, medium quality


In [3]:
silver = bronze.copy()

## missing

In [5]:
silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1000
NOME_MEDICAMENTO_WHODRUG                          5159
PRINCIPIOS_ATIVOS_WHODRUG                         5649
CODIGO_ATC                                        5159
DETENTOR_REGISTRO                               426201
CONCENTRACAO                                    460259
COMPONENTE_SUSPEITO                             496737
ACAO_ADOTADA                                    267347
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    534294
INDICACAO_MEDDRA                                325534
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          315982
DOSE                                            339554
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       275637
DURACAO                                         473049
INICIO_ADMINISTRACAO                            288439
FIM_ADMINISTRACAO                               395839
FORMA_FARM

In [6]:
for col in silver.select_dtypes(include=['object']).columns:
    silver[col] = normalize_rows(silver[col])

In [7]:
silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1000
NOME_MEDICAMENTO_WHODRUG                          5159
PRINCIPIOS_ATIVOS_WHODRUG                         5649
CODIGO_ATC                                        5159
DETENTOR_REGISTRO                               426201
CONCENTRACAO                                    460259
COMPONENTE_SUSPEITO                             496737
ACAO_ADOTADA                                    267347
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    534294
INDICACAO_MEDDRA                                325534
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          315982
DOSE                                            339554
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       275637
DURACAO                                         473049
INICIO_ADMINISTRACAO                            288439
FIM_ADMINISTRACAO                               395839
FORMA_FARM

## datas - 'INICIO_ADMINISTRACAO', 'FIM_ADMINISTRACAO'



In [8]:
colunas_data = ["INICIO_ADMINISTRACAO", "FIM_ADMINISTRACAO"]

In [9]:
silver[colunas_data].info()
silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   INICIO_ADMINISTRACAO  264448 non-null  object
 1   FIM_ADMINISTRACAO     157048 non-null  object
dtypes: object(2)
memory usage: 8.4+ MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,20181124,NaN
1,20181122,20181122
2,20181103,20181115
3,20181016,20181115
4,20181024,NaN


In [10]:
for col in colunas_data:
    if col in silver.columns:
        silver[col] = normalize_date_column(silver[col])

In [11]:
silver[colunas_data].info()
silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   INICIO_ADMINISTRACAO  220969 non-null  datetime64[ns]
 1   FIM_ADMINISTRACAO     142767 non-null  datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 8.4 MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,2018-11-24,NaT
1,2018-11-22,2018-11-22
2,2018-11-03,2018-11-15
3,2018-10-16,2018-11-15
4,2018-10-24,NaT


## detentor_registro

In [12]:
hist_dim_detentor_registro = pd.read_parquet(Path(project_root) / "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro.parquet")
hist_dim_detentor_registro.head()

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_PADRONIZADO,DETENTOR_REGISTRO_CHAVE,DETENTOR_REGISTRO_VALOR
0,ABACUS MEDICINE,ABACUS MEDICINE,1,ABACUS MEDICINE
1,ABBOLT,ABBOT,2,ABBOT
2,ABBOT,ABBOT,2,ABBOT
3,ABBOT BRASIL,ABBOT,2,ABBOT
4,ABBOT L: 0071/20 V: 31/01/2022,ABBOT,2,ABBOT


In [13]:
hist_dim_detentor_registro.columns

Index(['DETENTOR_REGISTRO', 'DETENTOR_REGISTRO_PADRONIZADO',
       'DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR'],
      dtype='object')

In [ ]:
# RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1,
    "CONCOMITANTE": 2,
    "MEDICAMENTO NAO ADMINISTRADO": 3,
    "INTERACAO": 4,
}
# COMPONENTE_SUSPEITO
componente_suspeito_map = {
    "PRINCIPIO ATIVO": 1, "EXCIPIENTE, NAO CLASSIFICADO": 2, "SOLVENTE": 3, "CORANTE": 4,
    "CONSERVANTE": 5, "AGENTE FLAVORIZADOR": 6, "EXCESSO PERCENTUAL": 7, "ANTIOXIDANTE": 8,
    "ESTABILIZANTE": 9
}

# ACAO_ADOTADA
acao_adotada_map = {
    "RETIRADA DO MEDICAMENTO": 1, "SEM ALTERACAO DA DOSE": 2, "NAO APLICAVEL": 3,
    "REDUCAO DA DOSE": 4, "AUMENTO DA DOSE": 5
}

# VIA_ADMINISTRACAO_MAE_PAI
via_administracao_mae_pai_map = {
    "ORAL": 1, "INTRAVENOSA (IV)": 2, "SUBCUTANEA (SC)": 3, "INTRAMUSCULAR (IM)": 4,
    "INTRACEREBRAL": 5, "RESPIRATORIA": 6, "INTRAUTERINA": 7
}
## dfs
hist_dim_detentor_registro = pd.read_parquet(
    Path(project_root) / "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro.parquet"
)

In [18]:
mappings_config = [
    {
        "kind": "dict",
        "col": "RELACAO_MEDICAMENTO_EVENTO",
        "map": relacao_medicamento_evento_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    },
    {
        "kind": "dict",
        "col": "COMPONENTE_SUSPEITO",
        "map": componente_suspeito_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    },
    {
        "kind": "dict",
        "col": "ACAO_ADOTADA",
        "map": acao_adotada_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    },
    {
        "kind": "dict",
        "col": "VIA_ADMINISTRACAO_MAE_PAI",
        "map": via_administracao_mae_pai_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    },
    {
        "kind": "df",
        "col": "DETENTOR_REGISTRO",
        "dim_df": hist_dim_detentor_registro,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": True,
    }, 
]

NameError: name 'componente_suspeito_map' is not defined

In [16]:
silver = apply_mappings(silver, mappings_config)
silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552887 entries, 0 to 552886
Data columns (total 25 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                     552887 non-null  object        
 1   NOME_MEDICAMENTO_WHODRUG                      547728 non-null  object        
 2   PRINCIPIOS_ATIVOS_WHODRUG                     547238 non-null  object        
 3   CODIGO_ATC                                    547728 non-null  object        
 4   CONCENTRACAO                                  92628 non-null   object        
 5   COMPONENTE_SUSPEITO                           56150 non-null   object        
 6   ACAO_ADOTADA                                  285540 non-null  object        
 7   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  18593 non-null   object        
 8   INDICACAO_MEDDRA                              227353 n

# PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO

In [4]:
col = "PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO"

# ... TODA sua limpeza anterior ...
# e terminou com:
bronze[col] = bronze[col].apply(
    lambda x: [i.strip() for i in x.split("|")] if x else []
)


In [5]:
bronze.head()

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
0,BR-ANVISA-300000004,Suspeito,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,None,None,[],None,None,250 milligram (mg),6 hora,None,4 dia,20181124,None,None,infusão intravenosa,None,5833018,2025-11-17
1,BR-ANVISA-300000005,Suspeito,Paracemol,Paracetamol,N02BE,None,None,None,Retirada do medicamento,[],None,None,None,None,None,None,20181122,20181122,None,oral,None,None,2025-11-17
2,BR-ANVISA-300000007,Suspeito,Diamox,Acetazolamide sodium,C03|N03AX|S01EC,None,None,None,Retirada do medicamento,[],None,None,250 milligram (mg),6 hora,250mg a cada 6 horas,None,20181103,20181115,None,oral,None,None,2025-11-17
3,BR-ANVISA-300000007,Suspeito,Hidantal,Phenytoin,N03AB,None,None,None,Retirada do medicamento,[],None,None,None,None,100mg a cada 8 horas,None,20181016,20181115,None,oral,None,None,2025-11-17
4,BR-ANVISA-300000008,Suspeito,Nitroglicerina,Glyceryl trinitrate,C01DA|C05AE,Cristália,None,None,None,[],None,None,None,None,None,None,20181024,None,None,infusão intravenosa,None,None,2025-11-17


In [12]:
import re

col = "PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO"

def clean_problema_token(p: str) -> str:
    p = str(p).upper()
    # remove o marcador X000D onde quer que esteja
    p = p.replace("X000D", " ")
    # normaliza separadores (underscore, espaço múltiplo etc.)
    p = re.sub(r"[_\s]+", " ", p)
    p = p.strip()
    # descarta lixo puro
    if p in ("", "X000D"):
        return ""
    return p

# 1) limpar string "bruta"
s = bronze[col].fillna("").astype(str).str.upper()

s = s.str.replace(r"X000D", " ", regex=True)
s = s.str.replace(r'[\r\n\t]', " ", regex=True)
s = s.str.replace('"', "")

# normaliza separador principal como "|"
s = s.str.replace(r"[|_]+", "|", regex=True)
s = s.str.replace(r"\|{2,}", "|", regex=True)
s = s.str.strip(" |")

# 2) virar LISTA já com cada item limpo
bronze[col] = s.apply(
    lambda txt: [
        clean_problema_token(p)
        for p in txt.split("|")
        if clean_problema_token(p)  # só mantém se não for vazio/lixo
    ]
)

In [13]:
bronze[col].head().to_list()


[[], [], [], [], []]

In [14]:
import re
from unidecode import unidecode
import pandas as pd

def _slug_col_name(texto: str) -> str:
    """
    Normaliza texto para virar nome de coluna:
    - remove acentos
    - coloca maiúsculas
    - troca não-alfanumérico por "_"
    - colapsa múltiplos "_" em um só
    - tira "_" das pontas
    """
    s = unidecode(str(texto)).upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")
    return s


def expandir_lista_wide(df, col, prefix=None, dtype="int8"):
    """
    Cria colunas dummies (0/1) para cada valor distinto em uma coluna de LISTAS.

    Ex:
      df[col] = ['ABUSO', 'ERRO DE MEDICAÇÃO']
      -> GRUPO_PROB_ABUSO = 1, GRUPO_PROB_ERRO_DE_MEDICACAO = 1

    Parâmetros
    ----------
    df : pd.DataFrame
        DataFrame com a coluna de listas.
    col : str
        Nome da coluna que contém listas (ex: PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO).
    prefix : str, opcional
        Prefixo para os nomes das colunas geradas. Default: f"{col}_" normalizado.
    dtype : str, default "int8"
        Tipo para as colunas 0/1.

    Retorna
    -------
    df : pd.DataFrame
        Mesmo DataFrame de entrada, com novas colunas 0/1.
    """
    if col not in df.columns:
        return df

    # garante prefixo bonitinho
    if prefix is None:
        prefix = _slug_col_name(col) + "_"

    s = df[col]

    # explode para descobrir todos os valores distintos
    categorias = (
        s.explode()
         .dropna()
         .astype(str)
         .str.strip()
         .loc[lambda x: x != ""]
         .drop_duplicates()
         .sort_values()
    )

    for valor in categorias:
        col_out = prefix + _slug_col_name(valor)
        # cria dummy: 1 se o valor estiver na lista daquela linha
        df[col_out] = s.apply(
            lambda lst: valor in lst if isinstance(lst, (list, tuple, set)) else False
        ).astype(dtype)

    return df


In [15]:
bronze = expandir_lista_wide(
    bronze,
    col="PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO",
    prefix="PROB_ADIC_",   # se quiser um prefixo mais curto/bonito
)

bronze.filter(like="PROB_ADIC_").head()


,PROB_ADIC_ABUSO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0


bronze = expandir_lista_wide(
    bronze,
    col="PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO",
    prefix="PROB_ADIC_",   # se quiser um prefixo mais curto/bonito
)

bronze.filter(like="PROB_ADIC_").head()
